In [2]:
import pandas as pd
import os

print(pd.__version__)

2.3.1


In [4]:
gdsc = pd.read_excel("../data/raw/GDSC2_drug_response.xlsx")
print(gdsc.shape)
print(gdsc.columns.tolist())
print(gdsc.head(3))

(242036, 19)
['DATASET', 'NLME_RESULT_ID', 'NLME_CURVE_ID', 'COSMIC_ID', 'CELL_LINE_NAME', 'SANGER_MODEL_ID', 'TCGA_DESC', 'DRUG_ID', 'DRUG_NAME', 'PUTATIVE_TARGET', 'PATHWAY_NAME', 'COMPANY_ID', 'WEBRELEASE', 'MIN_CONC', 'MAX_CONC', 'LN_IC50', 'AUC', 'RMSE', 'Z_SCORE']
  DATASET  NLME_RESULT_ID  NLME_CURVE_ID  COSMIC_ID CELL_LINE_NAME  \
0   GDSC2             401       18945558     683667         PFSK-1   
1   GDSC2             401       18945796     684052           A673   
2   GDSC2             401       18946078     684057            ES5   

  SANGER_MODEL_ID     TCGA_DESC  DRUG_ID     DRUG_NAME PUTATIVE_TARGET  \
0       SIDM01132            MB     1003  Camptothecin            TOP1   
1       SIDM00848  UNCLASSIFIED     1003  Camptothecin            TOP1   
2       SIDM00263  UNCLASSIFIED     1003  Camptothecin            TOP1   

      PATHWAY_NAME  COMPANY_ID WEBRELEASE  MIN_CONC  MAX_CONC   LN_IC50  \
0  DNA replication        1046          Y    0.0001       0.1 -1.462148   
1

In [5]:
drug_names = gdsc["DRUG_NAME"].dropna().unique().tolist()
print(f"Unique drugs: {len(drug_names)}")
print(drug_names[:20])

Unique drugs: 286
['Camptothecin', 'Vinblastine', 'Cisplatin', 'Cytarabine', 'Docetaxel', 'Methotrexate', 'Tretinoin', 'Gefitinib', 'Navitoclax', 'Vorinostat', 'Nilotinib', 'Refametinib', 'Temsirolimus', 'Olaparib', 'Veliparib', 'Bosutinib', 'Lenalidomide', 'Axitinib', 'AZD7762', 'GW441756']


In [6]:
import requests
import time

def get_pubchem_data(name):
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
        f"{requests.utils.quote(name)}/property/"
        f"CanonicalSMILES,InChIKey/JSON"
    )
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            props = r.json()["PropertyTable"]["Properties"][0]
            return props.get("CanonicalSMILES"), props.get("InChIKey")
    except Exception:
        pass
    return None, None

results = []
failed = []

for i, name in enumerate(drug_names):
    smiles, inchikey = get_pubchem_data(name)
    results.append({
        "drug_name": name,
        "smiles": smiles,
        "inchikey": inchikey,
        "inchikey_connectivity": inchikey.split("-")[0] if inchikey else None
    })
    if smiles is None:
        failed.append(name)
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(drug_names)} done...")
    time.sleep(0.22)

drug_df = pd.DataFrame(results)
print(f"\nFound SMILES: {drug_df['smiles'].notna().sum()}/{len(drug_names)}")
print(f"Failed: {len(failed)}")
print(f"Failed drugs: {failed}")

  20/286 done...
  40/286 done...
  60/286 done...
  80/286 done...
  100/286 done...
  120/286 done...
  140/286 done...
  160/286 done...
  180/286 done...
  200/286 done...
  220/286 done...
  240/286 done...
  260/286 done...
  280/286 done...

Found SMILES: 0/286
Failed: 286
Failed drugs: ['Camptothecin', 'Vinblastine', 'Cisplatin', 'Cytarabine', 'Docetaxel', 'Methotrexate', 'Tretinoin', 'Gefitinib', 'Navitoclax', 'Vorinostat', 'Nilotinib', 'Refametinib', 'Temsirolimus', 'Olaparib', 'Veliparib', 'Bosutinib', 'Lenalidomide', 'Axitinib', 'AZD7762', 'GW441756', 'Lestaurtinib', 'SB216763', 'Tanespimycin', 'Motesanib', 'KU-55933', 'Elesclomol', 'Afatinib', 'Vismodegib', 'Staurosporine', 'PLX-4720', 'BX795', 'NU7441', 'SL0101', 'Doramapimod', 'JNK Inhibitor VIII', 'Wee1 Inhibitor', 'Nutlin-3a (-)', 'Mirin', 'PD173074', 'ZM447439', 'Alisertib', 'RO-3306', 'MK-2206', 'Palbociclib', 'Dactolisib', 'Pictilisib', 'AZD8055', 'PD0325901', 'SB590885', 'Selumetinib', 'CCT007093', 'Obatoclax Mesyl

In [8]:
def get_pubchem_data(name):
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
        f"{requests.utils.quote(name)}/property/"
        f"CanonicalSMILES,IsomericSMILES,InChIKey/JSON"
    )
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            props = r.json()["PropertyTable"]["Properties"][0]
            # grab whichever SMILES key is present
            smiles = (
                props.get("CanonicalSMILES") or
                props.get("IsomericSMILES") or
                props.get("ConnectivitySMILES")
            )
            return smiles, props.get("InChIKey")
    except Exception:
        pass
    return None, None

# Quick sanity check before the full run
smiles, ik = get_pubchem_data("Cisplatin")
print(f"Cisplatin: {smiles} | {ik}")

Cisplatin: N.N.Cl[Pt]Cl | LXZZYRPGZAFOLE-UHFFFAOYSA-L


In [9]:
results = []
failed = []

for i, name in enumerate(drug_names):
    smiles, inchikey = get_pubchem_data(name)
    results.append({
        "drug_name": name,
        "smiles": smiles,
        "inchikey": inchikey,
        "inchikey_connectivity": inchikey.split("-")[0] if inchikey else None
    })
    if smiles is None:
        failed.append(name)
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(drug_names)} done...")
    time.sleep(0.22)

drug_df = pd.DataFrame(results)
print(f"\nFound SMILES: {drug_df['smiles'].notna().sum()}/{len(drug_names)}")
print(f"Failed: {len(failed)}")
print(f"Failed drugs: {failed}")

  20/286 done...
  40/286 done...
  60/286 done...
  80/286 done...
  100/286 done...
  120/286 done...
  140/286 done...
  160/286 done...
  180/286 done...
  200/286 done...
  220/286 done...
  240/286 done...
  260/286 done...
  280/286 done...

Found SMILES: 229/286
Failed: 57
Failed drugs: ['Nutlin-3a (-)', 'Bleomycin (50 uM)', 'IAP_5620', 'BDOCA000347a', 'BDF00022089a', 'BDILV000379a', 'Picolinici-acid', 'CDK9_5576', 'CDK9_5038', 'Eg5_9814', 'ERK_2440', 'ERK_6604', 'IRAK4_4710', 'JAK1_8709', 'PAK_5339', 'TAF1_5496', 'ULK1_4989', 'VSP34_8731', 'IGF1R_3801', 'JAK_8517', 'GSK2256098C', 'GSK2276186C', 'GSK626616AC', 'GSK3337463A', 'GSK2830371A', 'LMB_AB1', 'LMB_AB2', 'LMB_AB3', '123829', '765771', '123138', '50869', '720427', '667880', '729189', '741909', '743380', '150412', '615590', '630600', '776928', 'KRAS (G12C) Inhibitor-12', 'BDP-00009066', 'BPD-00008900', 'N25720-51-A1', 'N27922-53-1', 'N30652-18-1', 'N29087-69-1', 'VTP-A', 'VTP-B', 'PBD-288', 'CT7033-2', 'GSK-LSD1-2HCl ', 'T

In [10]:
import gzip

# Load JUMP-CP compound metadata
jump = pd.read_csv("../data/raw/compound.csv.gz")  # adjust path if needed
jump = jump.dropna(subset=["Metadata_InChIKey"])

# Build lookup sets
jump_exact   = set(jump["Metadata_InChIKey"].tolist())
jump_connect = set(k.split("-")[0] for k in jump_exact)

# Match
exact_matches        = []
connectivity_matches = []
no_match             = []

for _, row in drug_df.dropna(subset=["inchikey"]).iterrows():
    name   = row["drug_name"]
    ik     = row["inchikey"]
    ik_con = row["inchikey_connectivity"]

    if ik in jump_exact:
        exact_matches.append(name)
    elif ik_con in jump_connect:
        connectivity_matches.append(name)
    else:
        no_match.append(name)

total_matched = len(exact_matches) + len(connectivity_matches)

print("="*50)
print("JUMP-CP × GDSC2 OVERLAP")
print("="*50)
print(f"GDSC2 drugs with InChIKey:    {len(drug_df.dropna(subset=['inchikey']))}")
print(f"Exact matches:                {len(exact_matches)}")
print(f"Connectivity matches:         {len(connectivity_matches)}")
print(f"TOTAL matched:                {total_matched} ({100*total_matched/229:.1f}%)")
print(f"No match:                     {len(no_match)}")
print(f"\nProjected data points:        {total_matched * 700:,}")
print(f"\nExact matches: {sorted(exact_matches)}")
print(f"\nConnectivity matches: {sorted(connectivity_matches)}")
print(f"\nNo match: {sorted(no_match)}")

JUMP-CP × GDSC2 OVERLAP
GDSC2 drugs with InChIKey:    229
Exact matches:                98
Connectivity matches:         77
TOTAL matched:                175 (76.4%)
No match:                     54

Projected data points:        122,500

Exact matches: ['5-Fluorouracil', 'A-366', 'AGI-5198', 'AGI-6780', 'AZD1332', 'AZD5438', 'Acetalax', 'BEN', 'BX795', 'Bicalutamide', 'Bosutinib', 'Buparlisib', 'CHIR-99021', 'CZC24832', 'Cediranib', 'Cyclophosphamide', 'Dabrafenib', 'Dasatinib', 'Elesclomol', 'Entinostat', 'Entospletinib', 'Erlotinib', 'Foretinib', 'GNE-317', 'GSK1904529A', 'GSK2578215A', 'GSK2606414', 'GSK269962A', 'GSK2801', 'GSK343', 'Gefitinib', 'I-BRD9', 'IOX2', 'IWP-2', 'JNK Inhibitor VIII', 'KU-55933', 'LGK974', 'LJI308', 'LY2109761', 'Lapatinib', 'Leflunomide', 'Lenalidomide', 'Linsitinib', 'Luminespib', 'MK-1775', 'MK-2206', 'ML323', 'MN-64', 'Motesanib', 'NU7441', 'NVP-ADW742', 'Nilotinib', 'OF-1', 'OSI-027', 'Olaparib', 'Osimertinib', 'P22077', 'PCI-34051', 'PD173074', 'PF-

In [11]:
# Save matched drugs
drug_df.to_csv("../data/processed/gdsc2_drugs_with_smiles.csv", index=False)

overlap_df = pd.DataFrame({
    "drug_name": exact_matches + connectivity_matches,
    "match_type": ["exact"]*len(exact_matches) + ["connectivity"]*len(connectivity_matches)
}).merge(drug_df[["drug_name", "smiles", "inchikey"]], on="drug_name", how="left")

overlap_df.to_csv("../data/processed/gdsc2_jump_overlap.csv", index=False)

print(f"Saved {len(overlap_df)} matched drugs to gdsc2_jump_overlap.csv")
print(overlap_df.head())

Saved 175 matched drugs to gdsc2_jump_overlap.csv
    drug_name match_type                                             smiles  \
0   Gefitinib      exact  COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OC...   
1  Vorinostat      exact                   C1=CC=C(C=C1)NC(=O)CCCCCCC(=O)NO   
2   Nilotinib      exact  CC1=C(C=C(C=C1)C(=O)NC2=CC(=CC(=C2)C(F)(F)F)N3...   
3    Olaparib      exact  C1CC1C(=O)N2CCN(CC2)C(=O)C3=C(C=CC(=C3)CC4=NNC...   
4   Bosutinib      exact  CN1CCN(CC1)CCCOC2=C(C=C3C(=C2)N=CC(=C3NC4=CC(=...   

                      inchikey  
0  XGALLCVXEZPNRQ-UHFFFAOYSA-N  
1  WAEXFXRVDQXREF-UHFFFAOYSA-N  
2  HHZIURLSWUIHRB-UHFFFAOYSA-N  
3  FDLYAMZZIXQODN-UHFFFAOYSA-N  
4  UBPYILGKFZZVDX-UHFFFAOYSA-N  
